# Buzzlytics — YOLOv8 Fine-Tuning (3-class: bee / pollen_bee / varroa_bee)

Fine-tunes **YOLOv8n** on the unified 3-class bee dataset.

## One-time prep (build the dataset zip)
The dataset is built from two public sources by `datasets/prepare_dataset.py`
(VnPollenBee + VarroaDataset). Run it once, then zip + upload to Drive:

1. `python datasets/prepare_dataset.py` → writes `datasets/data/{train,valid,test}`.
2. Zip the `data` folder so the zip contains `train/ valid/ test/` (each with `images/` + `labels/`).
3. Upload `bee_dataset.zip` to **Google Drive** → Share → Anyone with the link → Copy link.

## Run
4. Runtime → Change runtime type → **GPU** (T4/L4/A100).
5. Paste the Drive link into `DATASET_URL` below.
6. Runtime → **Run all**.

**Class balance note:** pollen is sparse — expect lower mAP on `pollen_bee`; note it as a known limitation.

In [ ]:
# === 0. Configuration ===
# Paste your Google Drive SHARE LINK to bee_dataset.zip (Anyone-with-the-link).
# e.g. https://drive.google.com/file/d/1AbCdEfGhIjKlMnOpQ/view?usp=sharing
DATASET_URL = "PASTE_GDRIVE_SHARE_LINK_HERE"

EPOCHS = 100
IMGSZ  = 640
BATCH  = 16      # bump to 64 on L4/A100
MODEL  = "yolov8n.pt"   # nano: justified by the 10-FPS requirement

assert DATASET_URL != "PASTE_GDRIVE_SHARE_LINK_HERE", "Paste your Google Drive share link first"

In [ ]:
# === 1. Confirm GPU ===
import torch
print("CUDA available:", torch.cuda.is_available())
print("Device:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU (switch Runtime -> GPU!)")

In [ ]:
# === 2. Install dependencies ===
!pip -q install "ultralytics==8.4.71" gdown pyyaml

In [ ]:
# === 3. Download the dataset zip from Google Drive + extract ===
import gdown, zipfile
from pathlib import Path
from collections import Counter

gdown.download(url=DATASET_URL, output="/content/bee_dataset.zip", quiet=False, fuzzy=True)
with zipfile.ZipFile("/content/bee_dataset.zip") as z:
    z.extractall("/content/dataset")

# Auto-locate the dataset root (the dir that directly contains train/images).
DATA_DIR = None
for p in Path("/content/dataset").rglob("train/images"):
    DATA_DIR = str(p.parent.parent)
    break
assert DATA_DIR, "No train/images found in the zip. Re-zip so it contains train/ valid/ test/ each with images/ + labels/."
print("DATA_DIR:", DATA_DIR)

# Sanity check: class distribution (0=bee 1=pollen 2=varroa)
c = Counter()
for txt in Path(DATA_DIR).rglob("labels/*.txt"):
    for line in txt.read_text().splitlines():
        if line.strip():
            c[int(line.split()[0])] += 1
print("instances per class:", dict(sorted(c.items())))
for s in ["train", "valid", "test"]:
    d = Path(DATA_DIR) / s / "images"
    print(f"{s}: {len(list(d.glob('*'))) if d.is_dir() else 0} images")

In [ ]:
# === 4. Write the dataset yaml (absolute path to the extracted data) ===
import yaml
data_cfg = {
    "path": DATA_DIR,
    "train": "train/images",
    "val": "valid/images",
    "test": "test/images",
    "nc": 3,
    "names": {0: "bee", 1: "pollen_bee", 2: "varroa_bee"},
}
with open("/content/bee_dataset.yaml", "w") as f:
    yaml.safe_dump(data_cfg, f, sort_keys=False)
print(open("/content/bee_dataset.yaml").read())

In [ ]:
# === 5. Fine-tune YOLOv8n ===
from ultralytics import YOLO
model = YOLO(MODEL)
results = model.train(
    data="/content/bee_dataset.yaml",
    epochs=EPOCHS,
    imgsz=IMGSZ,
    batch=BATCH,
    name="bee_detector",
    patience=25,
    seed=42,
    hsv_h=0.015, hsv_s=0.7, hsv_v=0.4,
    fliplr=0.5, mosaic=1.0, degrees=5.0,
)

In [ ]:
# === 6. Validate + per-class metrics ===
best = "runs/detect/bee_detector/weights/best.pt"
metrics = YOLO(best).val(data="/content/bee_dataset.yaml", split="test")
print("\nmAP@50:     ", round(float(metrics.box.map50), 4))
print("mAP@50-95:  ", round(float(metrics.box.map), 4))
names = {0: "bee", 1: "pollen_bee", 2: "varroa_bee"}
print("\nPer-class mAP@50:")
for i, ap in zip(metrics.box.ap_class_index, metrics.box.ap50):
    print(f"  {names.get(int(i), i):<12} {float(ap):.4f}")

In [ ]:
# === 7. Download best.pt (place at cv_pipeline/weights/best.pt in your repo) ===
from google.colab import files
files.download("runs/detect/bee_detector/weights/best.pt")